# Lead-wise Transformer: Cross-lead ECG Classification

**Novel contribution:** After per-lead temporal encoding (shared weights), a cross-lead attention layer lets each lead attend to all other leads — mirroring how cardiologists compare leads in diagnosis (e.g. ST elevation in leads II, III, aVF simultaneously for inferior MI). All other models in this project are channel-independent; this is the first to model inter-lead relationships explicitly.

In [ ]:
# Cell 1: Setup
import sys, os, warnings
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import json
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config                import CFG
from src.preprocessing.label_utils   import load_all_labels
from src.preprocessing.dataset_full  import ECGDatasetFull
from src.models.leadwise_transformer import LeadwiseTransformer, build_leadwise_with_peft
from src.training.train_peft         import run_experiment

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

In [ ]:
# Cell 2: Load data
DATA_PATH = CFG['data']['path']
Y = load_all_labels(DATA_PATH + 'ptbxl_database.csv', DATA_PATH + 'scp_statements.csv')
train_df = Y[Y.strat_fold <  9]
val_df   = Y[Y.strat_fold == 9]
test_df  = Y[Y.strat_fold == 10]

train_ds = ECGDatasetFull(train_df, DATA_PATH)
val_ds   = ECGDatasetFull(val_df,   DATA_PATH)

print(f"Train: {len(train_df)} records  ({len(train_ds)} samples)")
print(f"Val:   {len(val_df)} records  ({len(val_ds)} samples)")
print(f"Test:  {len(test_df)} records (held out)")

In [ ]:
# Cell 3: Architecture inspection + PEFT verification (Step E)
_dummy = torch.randn(4, 12, 1000)

# Base model
base = LeadwiseTransformer()
bp = base.count_parameters()
with torch.no_grad():
    out = base(_dummy)
assert out.shape == (4, 5), f"Expected (4, 5), got {out.shape}"
print(f"Base LeadwiseTransformer")
print(f"  Trainable: {bp['trainable']:,} / {bp['total']:,} ({bp['percentage']})")
print(f"  Forward pass: {_dummy.shape} -> {out.shape} -- OK")
del base, out

# PEFT LoRA r=8 verification
lora8 = build_leadwise_with_peft(rank=8, use_dora=False)
p_lora8 = lora8.count_parameters()
pct_lora8 = p_lora8['trainable'] / p_lora8['total']
assert pct_lora8 < 0.10, f"LoRA trainable {pct_lora8:.1%} exceeds 10% — check target_modules"
with torch.no_grad():
    out = lora8(_dummy)
assert out.shape == (4, 5)
print(f"\nLoRA r=8  | Trainable: {p_lora8['trainable']:,} / {p_lora8['total']:,} ({p_lora8['percentage']}) -- OK")
del lora8, out

# PEFT DoRA r=8 verification
dora8 = build_leadwise_with_peft(rank=8, use_dora=True)
p_dora8 = dora8.count_parameters()
pct_dora8 = p_dora8['trainable'] / p_dora8['total']
assert pct_dora8 < 0.10
with torch.no_grad():
    out = dora8(_dummy)
assert out.shape == (4, 5)
print(f"DoRA r=8  | Trainable: {p_dora8['trainable']:,} / {p_dora8['total']:,} ({p_dora8['percentage']}) -- OK")
del dora8, out

del _dummy

In [ ]:
# Cell 4: Train all Lead-wise Transformer variants

# Exp 1 — LoRA r=8
model_1 = build_leadwise_with_peft(rank=8, use_dora=False)
auc_1, hist_lw_lora = run_experiment(
    model_1, train_ds, val_ds,
    experiment_name='leadwise_lora_r8',
    epochs=CFG['training']['epochs'],
    lr=CFG['training']['lr_peft'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results'],
)
del model_1; torch.cuda.empty_cache()

# Exp 2 — DoRA r=8
model_2 = build_leadwise_with_peft(rank=8, use_dora=True)
auc_2, hist_lw_dora = run_experiment(
    model_2, train_ds, val_ds,
    experiment_name='leadwise_dora_r8',
    epochs=CFG['training']['epochs'],
    lr=CFG['training']['lr_peft'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results'],
)
del model_2; torch.cuda.empty_cache()

# Exp 3 — LoRA r=4 (rank ablation)
model_3 = build_leadwise_with_peft(rank=4, use_dora=False)
auc_3, hist_lw_lora_r4 = run_experiment(
    model_3, train_ds, val_ds,
    experiment_name='leadwise_lora_r4',
    epochs=CFG['training']['epochs'],
    lr=CFG['training']['lr_peft'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results'],
)
del model_3; torch.cuda.empty_cache()

# Exp 4 — Full training, no PEFT (control)
model_4 = LeadwiseTransformer()
auc_4, hist_lw_full = run_experiment(
    model_4, train_ds, val_ds,
    experiment_name='leadwise_full',
    epochs=CFG['training']['epochs'],
    lr=CFG['training']['lr_peft'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results'],
)
del model_4; torch.cuda.empty_cache()

print(f"\nLeadwise summary:")
print(f"  LoRA r=8:       AUC {auc_1:.4f}")
print(f"  DoRA r=8:       AUC {auc_2:.4f}")
print(f"  LoRA r=4:       AUC {auc_3:.4f}")
print(f"  Full training:  AUC {auc_4:.4f}")

In [ ]:
# Cell 5: Load all experiment results from disk
RESULTS      = CFG['paths']['results']
SUPERCLASSES = CFG['data']['superclasses']

def load_hist(name):
    path = os.path.join(RESULTS, name, "history.json")
    if not os.path.exists(path):
        print(f"  Skipping {name} (no history.json)")
        return None
    with open(path) as f:
        return json.load(f)

# HuBERT variants
hist_hubert4 = load_hist('hubert_ecg_blocks4')
hist_hubert8 = load_hist('hubert_ecg_blocks8')
hist_lora    = load_hist('hubert_ecg_lora_r8')   # HuBERT LoRA
hist_dora    = load_hist('hubert_ecg_dora_r8')   # HuBERT DoRA

# Lead-wise variants
hist_lw_full    = load_hist('leadwise_full') or load_hist('leadwise_transformer')
hist_lw_lora    = load_hist('leadwise_lora_r8')
hist_lw_dora    = load_hist('leadwise_dora_r8')
hist_lw_lora_r4 = load_hist('leadwise_lora_r4')

# Baselines
with open(os.path.join(RESULTS, 'dummy_metrics.json')) as f:
    dummy_m = json.load(f)
with open(os.path.join(RESULTS, 'baseline_cnn', 'baseline_cnn_metrics.json')) as f:
    cnn_m = json.load(f)

print("HuBERT experiments:")
for name, h in [('4 blocks', hist_hubert4), ('8 blocks', hist_hubert8),
                ('LoRA r=8', hist_lora),    ('DoRA r=8', hist_dora)]:
    if h: print(f"  HuBERT {name}: best AUC={h['best_auc']:.4f} ({len(h['history'])} epochs)")

print("\nLead-wise experiments:")
for name, h in [('Full',    hist_lw_full),   ('LoRA r=8', hist_lw_lora),
                ('DoRA r=8', hist_lw_dora),  ('LoRA r=4', hist_lw_lora_r4)]:
    if h: print(f"  Leadwise {name}: best AUC={h['best_auc']:.4f} ({len(h['history'])} epochs)")

In [ ]:
# Cell 6: Full comparison table — all experiments, all metrics from JSON
def best_auc(h): return h['best_auc'] if h else float('nan')
def best_f1(h):
    if not h: return float('nan')
    return max(e["f1_macro"] for e in h["history"])

cnn_auc = cnn_m['auc_macro']

# Compute param counts for lightweight leadwise models inline
_lw_base = LeadwiseTransformer()
_lw_bp   = _lw_base.count_parameters(); del _lw_base
_lw_l8   = build_leadwise_with_peft(8, False)
_lw_l8p  = _lw_l8.count_parameters(); del _lw_l8
_lw_d8   = build_leadwise_with_peft(8, True)
_lw_d8p  = _lw_d8.count_parameters(); del _lw_d8
_lw_l4   = build_leadwise_with_peft(4, False)
_lw_l4p  = _lw_l4.count_parameters(); del _lw_l4

# HuBERT param counts from training output (loading model is too slow for results cell)
_HT = 93_323_397
KNOWN_PARAMS = {
    'hubert4':  f'{100*28_551_173/_HT:.0f}%',
    'hubert8':  f'{100*56_902_661/_HT:.0f}%',
    'hlora':    '~1%',
    'hdora':    '~1%',
}

def pct(n, d): return f'{100*n/d:.1f}%' if d else 'n/a'

rows = [
    {'Model': 'Dummy',          'Strategy': '--',              'AUC': dummy_m['auc_macro'], 'F1': dummy_m['f1_macro'], 'Params%': 'n/a'},
    {'Model': 'CNN',            'Strategy': 'Full training',   'AUC': cnn_auc,              'F1': cnn_m['f1_macro'],   'Params%': '<0.1%'},
]
if hist_hubert4:
    rows.append({'Model': 'HuBERT-ECG', 'Strategy': 'Unfreeze 4 blocks', 'AUC': best_auc(hist_hubert4),  'F1': best_f1(hist_hubert4),  'Params%': KNOWN_PARAMS['hubert4']})
if hist_hubert8:
    rows.append({'Model': 'HuBERT-ECG', 'Strategy': 'Unfreeze 8 blocks', 'AUC': best_auc(hist_hubert8),  'F1': best_f1(hist_hubert8),  'Params%': KNOWN_PARAMS['hubert8']})
if hist_lora:
    rows.append({'Model': 'HuBERT-ECG', 'Strategy': 'LoRA r=8',          'AUC': best_auc(hist_lora),     'F1': best_f1(hist_lora),     'Params%': KNOWN_PARAMS['hlora']})
if hist_dora:
    rows.append({'Model': 'HuBERT-ECG', 'Strategy': 'DoRA r=8',          'AUC': best_auc(hist_dora),     'F1': best_f1(hist_dora),     'Params%': KNOWN_PARAMS['hdora']})
if hist_lw_full:
    rows.append({'Model': 'Lead-wise',  'Strategy': 'Full training',      'AUC': best_auc(hist_lw_full),  'F1': best_f1(hist_lw_full),  'Params%': '100%'})
if hist_lw_lora:
    rows.append({'Model': 'Lead-wise',  'Strategy': 'LoRA r=8',           'AUC': best_auc(hist_lw_lora),  'F1': best_f1(hist_lw_lora),  'Params%': pct(_lw_l8p['trainable'], _lw_l8p['total'])})
if hist_lw_dora:
    rows.append({'Model': 'Lead-wise',  'Strategy': 'DoRA r=8',           'AUC': best_auc(hist_lw_dora),  'F1': best_f1(hist_lw_dora),  'Params%': pct(_lw_d8p['trainable'], _lw_d8p['total'])})
if hist_lw_lora_r4:
    rows.append({'Model': 'Lead-wise',  'Strategy': 'LoRA r=4',           'AUC': best_auc(hist_lw_lora_r4), 'F1': best_f1(hist_lw_lora_r4), 'Params%': pct(_lw_l4p['trainable'], _lw_l4p['total'])})

df = pd.DataFrame(rows)
df['vs CNN'] = (df['AUC'] - cnn_auc).map(lambda x: f'+{x:.4f}' if x > 0 else f'{x:.4f}')
print(df.to_string(index=False))

In [ ]:
# Cell 7: Learning curves — all experiments
fig, (ax_auc, ax_loss) = plt.subplots(1, 2, figsize=(16, 5))

experiments = []
if hist_hubert4:  experiments.append(("HuBERT 4 blocks",  hist_hubert4["history"],  "tab:blue",   "--"))
if hist_hubert8:  experiments.append(("HuBERT 8 blocks",  hist_hubert8["history"],  "tab:cyan",   "--"))
if hist_lora:     experiments.append(("HuBERT LoRA r=8",  hist_lora["history"],     "tab:blue",   "-"))
if hist_dora:     experiments.append(("HuBERT DoRA r=8",  hist_dora["history"],     "tab:purple", "-"))
if hist_lw_full:  experiments.append(("Leadwise full",    hist_lw_full["history"],  "tab:green",  "--"))
if hist_lw_lora:  experiments.append(("Leadwise LoRA r=8",hist_lw_lora["history"],  "tab:green",  "-"))
if hist_lw_dora:  experiments.append(("Leadwise DoRA r=8",hist_lw_dora["history"],  "tab:olive",  "-"))
if hist_lw_lora_r4: experiments.append(("Leadwise LoRA r=4",hist_lw_lora_r4["history"],"tab:lime","-"))

for name, h, color, ls in experiments:
    ep = [e["epoch"] for e in h]
    ax_auc.plot(ep,  [e["auc_macro"] for e in h], label=name, color=color, linestyle=ls, linewidth=2)
    ax_loss.plot(ep, [e["val_loss"]  for e in h], label=name, color=color, linestyle=ls, linewidth=2)

ax_auc.axhline(y=cnn_m['auc_macro'],   color='red',  linestyle=':', linewidth=1.5, label='CNN baseline')
ax_auc.axhline(y=dummy_m['auc_macro'], color='gray', linestyle=':', linewidth=1.0, label='Dummy')
ax_auc.set_title('Validation AUC over epochs')
ax_auc.set_xlabel('Epoch'); ax_auc.set_ylabel('AUC (macro)')
ax_auc.legend(fontsize=8); ax_auc.grid(True, alpha=0.3)

ax_loss.set_title('Validation loss over epochs')
ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
ax_loss.legend(fontsize=8); ax_loss.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs(CFG['paths']['figures'], exist_ok=True)
plt.savefig(CFG['paths']['figures'] + 'all_experiments_comparison.png', dpi=150)
plt.show()

In [ ]:
# Cell 8: Per-class AUC grouped bar chart — all experiments
fig, ax = plt.subplots(figsize=(14, 5))

plot_exps = []
for label, h in [("HuBERT 4b",    hist_hubert4),
                 ("HuBERT 8b",    hist_hubert8),
                 ("HuBERT LoRA",  hist_lora),
                 ("HuBERT DoRA",  hist_dora),
                 ("LW Full",      hist_lw_full),
                 ("LW LoRA r=8",  hist_lw_lora),
                 ("LW DoRA r=8",  hist_lw_dora),
                 ("LW LoRA r=4",  hist_lw_lora_r4)]:
    if h:
        best_ep = max(h["history"], key=lambda e: e["auc_macro"])
        plot_exps.append((label, best_ep["per_class"]))

n_exp = len(plot_exps)
n_cls = len(SUPERCLASSES)
x     = np.arange(n_cls)
width = 0.7 / n_exp

for i, (name, per_class) in enumerate(plot_exps):
    vals   = [per_class.get(cls, 0) for cls in SUPERCLASSES]
    offset = (i - (n_exp - 1) / 2) * width
    ax.bar(x + offset, vals, width, label=name)

ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, label='Chance')
ax.set_xticks(x); ax.set_xticklabels(SUPERCLASSES)
ax.set_ylim(0, 1.05)
ax.set_title('Per-class AUC at best epoch')
ax.set_xlabel('Superclass'); ax.set_ylabel('AUC')
ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(CFG['paths']['figures'] + 'per_class_auc_comparison.png', dpi=150)
plt.show()

In [ ]:
# Cell 9: Best model + parameter efficiency table
all_res = []
for name, h in [("HuBERT-ECG (4 blocks)", hist_hubert4),
                ("HuBERT-ECG (8 blocks)", hist_hubert8),
                ("HuBERT-ECG (LoRA r=8)", hist_lora),
                ("HuBERT-ECG (DoRA r=8)", hist_dora),
                ("Lead-wise full",         hist_lw_full),
                ("Lead-wise LoRA r=8",    hist_lw_lora),
                ("Lead-wise DoRA r=8",    hist_lw_dora),
                ("Lead-wise LoRA r=4",    hist_lw_lora_r4)]:
    if h:
        best_ep = max(h["history"], key=lambda e: e["auc_macro"])
        all_res.append((name, h["best_auc"], best_ep["per_class"]))

if all_res:
    best_name, best_auc_val, best_pc = max(all_res, key=lambda r: r[1])
    print(f"Best model: {best_name}  (AUC={best_auc_val:.4f})")
    print("Per-class AUC breakdown:")
    for cls in SUPERCLASSES:
        v = best_pc.get(cls, 0)
        print(f"  {cls:5s}: {v:.4f}  {'#' * int(v * 20)}")

# Full parameter efficiency table
from src.models.hubert_ecg_finetune import HuBERTECGClassifier, HuBERTECGPEFT

TOTAL_HUB = 93_323_397

print(f"\nParameter Efficiency")
print(f"  {'Method':<30s}  {'Trainable':>12s}  {'% total':>8s}")
print(f"  {'-'*55}")
print(f"  {'CNN (baseline)':<30s}  {'~50K':>12s}  {'<0.1%':>8s}")
print(f"  {'HuBERT full fine-tune':<30s}  {TOTAL_HUB:>12,}  {'100.0%':>8s}")

m4 = HuBERTECGClassifier(size='base', blocks_to_unfreeze=4)
p4 = m4.count_parameters(); del m4
print(f"  {'HuBERT selective 4b':<30s}  {p4['trainable']:>12,}  {100*p4['trainable']/TOTAL_HUB:>7.1f}%")

m8 = HuBERTECGClassifier(size='base', blocks_to_unfreeze=8)
p8 = m8.count_parameters(); del m8
print(f"  {'HuBERT selective 8b':<30s}  {p8['trainable']:>12,}  {100*p8['trainable']/TOTAL_HUB:>7.1f}%")

ml = HuBERTECGPEFT(rank=8, use_dora=False)
pl = ml.count_parameters(); del ml; torch.cuda.empty_cache()
print(f"  {'HuBERT LoRA r=8':<30s}  {pl['trainable']:>12,}  {100*pl['trainable']/pl['total']:>7.1f}%")

md = HuBERTECGPEFT(rank=8, use_dora=True)
pd_ = md.count_parameters(); del md; torch.cuda.empty_cache()
print(f"  {'HuBERT DoRA r=8':<30s}  {pd_['trainable']:>12,}  {100*pd_['trainable']/pd_['total']:>7.1f}%")

lw = LeadwiseTransformer(); plw = lw.count_parameters(); del lw
print(f"  {'Lead-wise full':<30s}  {plw['trainable']:>12,}  {'100.0%':>8s}")

lw8 = build_leadwise_with_peft(8, False); p8lw = lw8.count_parameters(); del lw8
print(f"  {'Lead-wise LoRA r=8':<30s}  {p8lw['trainable']:>12,}  {100*p8lw['trainable']/p8lw['total']:>7.1f}%")

lwd = build_leadwise_with_peft(8, True); pdlw = lwd.count_parameters(); del lwd
print(f"  {'Lead-wise DoRA r=8':<30s}  {pdlw['trainable']:>12,}  {100*pdlw['trainable']/pdlw['total']:>7.1f}%")

lw4 = build_leadwise_with_peft(4, False); p4lw = lw4.count_parameters(); del lw4
print(f"  {'Lead-wise LoRA r=4':<30s}  {p4lw['trainable']:>12,}  {100*p4lw['trainable']/p4lw['total']:>7.1f}%")

In [ ]:
# Cell 10: Efficiency vs AUC scatter plot — the paper's key figure
# X: trainable parameters (log scale)
# Y: macro AUC
# Color: blue=HuBERT, green=Lead-wise, red=CNN, gray=Dummy

fig, ax = plt.subplots(figsize=(10, 6))

# Compute leadwise param counts
_lw_base = LeadwiseTransformer(); _lw_bp = _lw_base.count_parameters(); del _lw_base
_lw_l8 = build_leadwise_with_peft(8, False); _lw_l8p = _lw_l8.count_parameters(); del _lw_l8
_lw_d8 = build_leadwise_with_peft(8, True);  _lw_d8p = _lw_d8.count_parameters(); del _lw_d8
_lw_l4 = build_leadwise_with_peft(4, False); _lw_l4p = _lw_l4.count_parameters(); del _lw_l4

# (label, trainable_params, auc, color, marker)
points = [
    ('Dummy',            None,              dummy_m['auc_macro'],         'gray',       'x'),
    ('CNN',              50_000,            cnn_m['auc_macro'],            'red',        'o'),
]
if hist_hubert4:  points.append(('HuBERT 4 blocks',  28_551_173,          best_auc(hist_hubert4),  'tab:blue',   'o'))
if hist_hubert8:  points.append(('HuBERT 8 blocks',  56_902_661,          best_auc(hist_hubert8),  'tab:blue',   's'))
if hist_lora:     points.append(('HuBERT LoRA r=8',  789_509,             best_auc(hist_lora),     'tab:blue',   '^'))
if hist_dora:     points.append(('HuBERT DoRA r=8',  826_373,             best_auc(hist_dora),     'cornflowerblue', 'D'))
if hist_lw_full:  points.append(('Leadwise full',    _lw_bp['trainable'], best_auc(hist_lw_full),  'tab:green',  'o'))
if hist_lw_lora:  points.append(('Leadwise LoRA r=8',_lw_l8p['trainable'],best_auc(hist_lw_lora),  'tab:green',  '^'))
if hist_lw_dora:  points.append(('Leadwise DoRA r=8',_lw_d8p['trainable'],best_auc(hist_lw_dora),  'limegreen',  'D'))
if hist_lw_lora_r4: points.append(('Leadwise LoRA r=4',_lw_l4p['trainable'],best_auc(hist_lw_lora_r4),'tab:green','v'))

for label, params, auc, color, marker in points:
    if params is None:
        continue
    ax.scatter(params, auc, color=color, marker=marker, s=120, zorder=3)
    ax.annotate(label, (params, auc),
                textcoords="offset points", xytext=(6, 4),
                fontsize=8, color=color)

ax.axhline(y=cnn_m['auc_macro'], color='red', linestyle=':', linewidth=1, alpha=0.5)
ax.set_xscale('log')
ax.set_xlabel('Trainable parameters (log scale)')
ax.set_ylabel('Macro AUC')
ax.set_title('Parameter Efficiency vs Performance\nBlue=HuBERT | Green=Lead-wise | Red=CNN')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CFG['paths']['figures'] + 'efficiency_scatter.png', dpi=150)
plt.show()

In [ ]:
# Parameter efficiency table 
from src.models.hubert_ecg_finetune import HuBERTECGClassifier, HuBERTECGPEFT

TOTAL = 93_323_397   # HuBERT-ECG-base confirmed total params

print(f"\nParameter Efficiency Comparison")
print(f"  {'Method':<28s}  {'Trainable':>14s}  {'% of total':>10s}")
print(f"  {'-'*58}")
print(f"  {'Full fine-tune':<28s}  {TOTAL:>14,}  {100.0:>9.1f}%")
print(f"  {'CNN (baseline)':<28s}  {'~50K':>14s}  {'<0.1%':>10s}")

m4 = HuBERTECGClassifier(size='base', blocks_to_unfreeze=4)
p4 = m4.count_parameters()
print(f"  {'Selective (4 blocks)':<28s}  {p4['trainable']:>14,}  {100*p4['trainable']/TOTAL:>9.1f}%")
del m4

m8 = HuBERTECGClassifier(size='base', blocks_to_unfreeze=8)
p8 = m8.count_parameters()
print(f"  {'Selective (8 blocks)':<28s}  {p8['trainable']:>14,}  {100*p8['trainable']/TOTAL:>9.1f}%")
del m8

ml = HuBERTECGPEFT(rank=8, use_dora=False)
pl = ml.count_parameters()
print(f"  {'LoRA r=8':<28s}  {pl['trainable']:>14,}  {100*pl['trainable']/pl['total']:>9.1f}%")
del ml; torch.cuda.empty_cache()

md = HuBERTECGPEFT(rank=8, use_dora=True)
pd_ = md.count_parameters()
print(f"  {'DoRA r=8':<28s}  {pd_['trainable']:>14,}  {100*pd_['trainable']/pd_['total']:>9.1f}%")
del md; torch.cuda.empty_cache()

lw = LeadwiseTransformer()
pl2 = lw.count_parameters()
print(f"  {'Lead-wise Transformer':<28s}  {pl2['trainable']:>14,}  {'100% (scratch)':>10s}")
del lw